# Session 2: translate a field, estimate the motion, score the forecast
**A. Real-world problem.** A radar picture is a 100 x 100 grid of rain rates. To nowcast, we
(1) estimate how the whole picture moved between two frames, (2) slide the latest frame
along that motion, and (3) score the result against what really happened. The motion is
rarely a whole number of pixels, so we must also shift by fractions of a pixel, and that has a cost.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
N = 100               # grid size (pixels); 1 pixel = 1 km
DT_MIN = 5.0          # minutes between frames
THR = 5.0             # rain-rate threshold (mm/h) used for yes/no scores

## Step 1: a synthetic radar field
A few Gaussian rain cells: each is (cx, cy, sigma, peak) with the centre in pixels, width in
pixels and centre rain rate in mm/h. The field is the sum of its cells.

In [ ]:
def make_field(cells, n=N):
    y, x = np.mgrid[0:n, 0:n]
    f = np.zeros((n, n))
    for cx, cy, sigma, peak in cells:
        f += peak * np.exp(-((x - cx) ** 2 + (y - cy) ** 2) / (2 * sigma ** 2))
    return f


def moved_cells(cells, u, v, frames):
    """The same cells after `frames` frames, moving (u, v) pixels per frame."""
    return [(cx + u * frames, cy + v * frames, s, p) for cx, cy, s, p in cells]

## Step 2: translate a field
*Integer shift*: copy the array to a new place and fill the gap with zero rain (rain that
leaves the picture is lost). *Sub-pixel shift*: the new value at pixel x is read at x - dx.
When x - dx falls between pixels, take a weighted average of the 4 neighbours (bilinear
interpolation). The weights add up to 1, so rain is not created or destroyed, but the
averaging blurs the field.

In [ ]:
def shift_int(field, dx, dy):
    """Integer shift: dx pixels east, dy pixels north. Rain pushed off the edge is lost."""
    ny, nx = field.shape
    out = np.zeros_like(field)
    xs, xe = max(0, dx), min(nx, nx + dx)
    ys, ye = max(0, dy), min(ny, ny + dy)
    if xe > xs and ye > ys:
        out[ys:ye, xs:xe] = field[ys - dy:ye - dy, xs - dx:xe - dx]
    return out


def shift_bilinear(field, dx, dy):
    """Shift by any real dx, dy (pixels) using bilinear interpolation; zero rain outside the picture."""
    ny, nx = field.shape
    y, x = np.mgrid[0:ny, 0:nx].astype(float)
    sx, sy = x - dx, y - dy
    x0, y0 = np.floor(sx).astype(int), np.floor(sy).astype(int)
    fx, fy = sx - x0, sy - y0
    pad = np.pad(field, 1)

    def at(yy, xx):
        ok = (yy >= -1) & (yy <= ny) & (xx >= -1) & (xx <= nx)
        return np.where(ok, pad[np.clip(yy + 1, 0, ny + 1), np.clip(xx + 1, 0, nx + 1)], 0.0)

    return ((1 - fx) * (1 - fy) * at(y0, x0) + fx * (1 - fy) * at(y0, x0 + 1)
            + (1 - fx) * fy * at(y0 + 1, x0) + fx * fy * at(y0 + 1, x0 + 1))


def x_variance(field):
    """Spread of the rain along x, in pixels squared (rain-weighted variance of the column index)."""
    x = np.arange(field.shape[1])
    col = field.sum(axis=0)
    m = (col * x).sum() / col.sum()
    return float((col * (x - m) ** 2).sum() / col.sum())

## Step 3: estimate the motion between two frames
Try every whole-pixel shift of the old frame and keep the one whose squared difference from the
new frame is smallest (brute-force search; it is the same idea as finding the peak of the
cross-correlation). Then refine to fractions of a pixel by searching a finer grid around it.

In [ ]:
def sse_for_shift(f0, f1, dx, dy):
    return float(((shift_bilinear(f0, dx, dy) - f1) ** 2).sum())


def estimate_shift(f0, f1, max_shift=8, step=0.1):
    """Return (dx, dy) in pixels per frame that best moves f0 onto f1."""
    best = None
    for dy in range(-max_shift, max_shift + 1):
        for dx in range(-max_shift, max_shift + 1):
            e = sse_for_shift(f0, f1, dx, dy)
            if best is None or e < best[0]:
                best = (e, dx, dy)
    _, bx, by = best
    offs = np.arange(-1.0, 1.0 + 1e-9, step)
    best = None
    for oy in offs:
        for ox in offs:
            e = sse_for_shift(f0, f1, bx + ox, by + oy)
            if best is None or e < best[0]:
                best = (e, bx + ox, by + oy)
    return float(best[1]), float(best[2])


def xcorr_peak(f0, f1):
    """Whole-pixel shift (dx, dy) at the peak of the circular cross-correlation (via FFT)."""
    c = np.fft.ifft2(np.conj(np.fft.fft2(f0)) * np.fft.fft2(f1)).real
    iy, ix = np.unravel_index(np.argmax(c), c.shape)
    ny, nx = c.shape
    return int(ix if ix <= nx // 2 else ix - nx), int(iy if iy <= ny // 2 else iy - ny)

## Step 4: forecasts and scores
*Persistence*: say nothing changes. *Advection*: move the latest frame by (shift per frame) x
(lead in frames). Scores compare forecast f with observation o:
* RMSE: square root of the mean squared difference (mm/h); 0 is perfect.
* CSI (critical success index): count pixels at or above a threshold. hits / (hits + misses + false alarms); 1 is perfect.
* FSS (fractions skill score): compare the *fraction* of rainy pixels in n x n windows. It
  forgives small position errors; the larger n, the more forgiving.

In [ ]:
def advect_forecast(latest, u, v, lead_frames):
    return shift_bilinear(latest, u * lead_frames, v * lead_frames)


def rmse(f, o):
    return float(np.sqrt(np.mean((f - o) ** 2)))


def csi_counts(hits, misses, false_alarms):
    d = hits + misses + false_alarms
    return float("nan") if d == 0 else hits / d


def csi(f, o, thr=THR):
    a, b = f >= thr, o >= thr
    return csi_counts(int((a & b).sum()), int((~a & b).sum()), int((a & ~b).sum()))


def box_mean(a, n):
    """Mean of a over n x n windows (n odd), zero outside the picture."""
    r = n // 2
    p = np.pad(a.astype(float), r + 1)
    c = p.cumsum(0).cumsum(1)
    ny, nx = a.shape
    s = c[n:n + ny, n:n + nx] - c[0:ny, n:n + nx] - c[n:n + ny, 0:nx] + c[0:ny, 0:nx]
    return s / (n * n)


def fss(f, o, thr=THR, n=1):
    pf, po = box_mean(f >= thr, n), box_mean(o >= thr, n)
    den = (pf ** 2).mean() + (po ** 2).mean()
    return float("nan") if den == 0 else float(1 - ((pf - po) ** 2).mean() / den)

## Step 5: a clean model for skill: two discs
Where the field is above the threshold, a Gaussian cell is a disc of radius
r = sigma * sqrt(2 ln(peak/threshold)). If the forecast disc is displaced from the true disc by
d, the CSI is (overlap area) / (union area). This explains why skill falls with lead time:
the displacement d grows like (wind error) x (lead time).

In [ ]:
def disc_csi(r, d):
    """CSI of two discs of radius r (km) whose centres are d (km) apart."""
    if d >= 2 * r:
        return 0.0
    ov = 2 * r * r * np.arccos(d / (2 * r)) - 0.5 * d * np.sqrt(4 * r * r - d * d)
    return float(ov / (2 * np.pi * r * r - ov))


def edge_radius(sigma, peak, thr=THR):
    return 0.0 if peak <= thr else float(sigma * np.sqrt(2 * np.log(peak / thr)))

In [ ]:
CELLS = [(15, 20, 5.0, 30.0), (25, 55, 7.0, 20.0), (40, 30, 4.0, 40.0)]
TRUE_SHIFT = (2.4, 1.3)        # pixels per frame (east, north)

if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    f0 = make_field(moved_cells(CELLS, *TRUE_SHIFT, -1))
    f1 = make_field(CELLS)
    obs60 = make_field(moved_cells(CELLS, *TRUE_SHIFT, 12))
    ex, ey = estimate_shift(f0, f1)
    print(f"estimated shift ({ex:.2f}, {ey:.2f}) px/frame, true {TRUE_SHIFT}")
    pers = f1
    adv = advect_forecast(f1, ex, ey, 12)
    print(f"60 min: persistence RMSE {rmse(pers, obs60):.2f} CSI {csi(pers, obs60):.3f} | "
          f"advection RMSE {rmse(adv, obs60):.2f} CSI {csi(adv, obs60):.3f}")
    fig, axs = plt.subplots(1, 4, figsize=(12, 3.2))
    for ax, fld, ttl in zip(axs, (f1, obs60, pers, adv), ("now", "actual at +60 min", "persistence", "advection")):
        ax.imshow(fld, origin="lower", cmap="Blues", vmin=0, vmax=40)
        ax.contour(obs60, levels=[THR], colors="r", linewidths=1)
        ax.set_title(ttl)
        ax.set_xlabel("x (km)")
    axs[0].set_ylabel("y (km)")
    fig.suptitle("Red line: where rain is at least 5 mm/h at +60 min")
    fig.tight_layout()
    fig.savefig(OUT / "forecast_panels.png", dpi=110)

    leads = np.arange(1, 13)
    cp, ca, rp, ra = [], [], [], []
    for k in leads:
        ob = make_field(moved_cells(CELLS, *TRUE_SHIFT, k))
        a = advect_forecast(f1, ex, ey, k)
        cp.append(csi(f1, ob))
        ca.append(csi(a, ob))
        rp.append(rmse(f1, ob))
        ra.append(rmse(a, ob))
    fig2, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    a1.plot(leads * DT_MIN, cp, "o-", label="persistence")
    a1.plot(leads * DT_MIN, ca, "s-", label="advection")
    a1.set_xlabel("lead time (min)")
    a1.set_ylabel("CSI at 5 mm/h")
    a1.legend()
    a1.grid(alpha=0.3)
    a2.plot(leads * DT_MIN, rp, "o-", label="persistence")
    a2.plot(leads * DT_MIN, ra, "s-", label="advection")
    a2.set_xlabel("lead time (min)")
    a2.set_ylabel("RMSE (mm/h)")
    a2.legend()
    a2.grid(alpha=0.3)
    fig2.suptitle("Skill against lead time for a translating field")
    fig2.tight_layout()
    fig2.savefig(OUT / "skill_vs_lead.png", dpi=110)

    one = make_field([(50, 50, 3.0, 30.0)])
    fr = np.linspace(0, 1, 21)
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    ax3.plot(fr, [shift_bilinear(one, f, 0).max() / one.max() for f in fr])
    ax3.set_xlabel("fractional shift (pixels)")
    ax3.set_ylabel("peak after shift / peak before")
    ax3.set_title("Bilinear interpolation blurs most at half a pixel")
    ax3.grid(alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(OUT / "interpolation_blur.png", dpi=110)
    print(f"half-pixel shift keeps {shift_bilinear(one, 0.5, 0).max() / one.max():.3f} of the peak")

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** Change `TRUE_SHIFT` in `lesson.py` to (-1.7, 0.6) and check that `estimate_shift` recovers it. Predict the 60-minute displacement in km first.
2. **Stretch.** Write `half_pixel_peak(sigma)` that returns the fraction of the peak that survives a half-pixel shift for a cell of width sigma, and print it for sigma 1, 2 and 3. Which cells suffer most?
3. **Explain.** Explain, with the two-disc model, why a forecast with a 5 km displacement can still be useful for a big cell but useless for a small one.

## Assessment
- **A. Conceptual.** Why does shifting a field by half a pixel make the peak lower, even though no rain is lost?
- **B. Numerical.** The estimated shift is (1.5, 0.5) pixels per 5 minutes. Give the wind components and speed in km/h and the displacement after 60 minutes.
- **C. Graph and simulation interpretation.** On the skill-against-lead-time graph for a translating field, which method wins and why does the other curve fall? Quote the 60 minute CSI values.
- **D. Design and optimisation challenge.** Design: a cell has a 5 mm/h radius of 9.47 km. You want a CSI of at least 0.5 at 60 minutes. How accurate must the wind be? What if you only need 30 minutes?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    cell = L.make_field([(50, 50, 3.0, 30.0)])
    checks.append(("a cell has its peak (30 mm/h) at its centre pixel", abs(cell[50, 50] - 30.0) < 1e-9 and cell.shape == (100, 100)))

    # ---- shifting
    si = L.shift_int(cell, 5, -3)
    checks.append(("integer shift moves the peak to pixel (x=55, y=47) and keeps the peak value",
                   abs(si[47, 55] - 30.0) < 1e-9 and abs(si.max() - 30.0) < 1e-9))
    checks.append(("rain pushed off the picture is lost by an integer shift", L.shift_int(cell, 100, 0).sum() == 0.0 and L.shift_int(cell, 60, 0).sum() < 0.01 * cell.sum()))
    checks.append(("a bilinear shift by whole pixels equals the integer shift",
                   np.allclose(L.shift_bilinear(cell, 5.0, -3.0), si, atol=1e-12)))
    half = L.shift_bilinear(cell, 0.5, 0.0)
    checks.append(("interpolation conserves total rain (shift by 5.5 pixels)",
                   abs(L.shift_bilinear(cell, 5.5, 0.0).sum() / cell.sum() - 1.0) < 1e-12))
    checks.append(("a half-pixel shift keeps 0.973 of the peak of a sigma = 3 cell (smoothing)",
                   abs(half.max() / cell.max() - 0.973) < 0.001))
    gain = L.x_variance(half) - L.x_variance(cell)
    checks.append(("hand calculation: the added blur is f(1-f) = 0.25 pixel^2 for f = 0.5",
                   abs(gain - 0.25) < 1e-3))
    g25 = L.x_variance(L.shift_bilinear(cell, 0.25, 0.0)) - L.x_variance(cell)
    checks.append(("the blur at f = 0.25 is 0.1875 and at f = 0.75 is the same", abs(g25 - 0.1875) < 1e-3
                   and abs(L.x_variance(L.shift_bilinear(cell, 0.75, 0.0)) - L.x_variance(cell) - 0.1875) < 1e-3))
    checks.append(("a shift of 0 changes nothing", np.allclose(L.shift_bilinear(cell, 0.0, 0.0), cell, atol=1e-12)))

    # ---- motion estimation
    f0 = L.make_field(L.CELLS)
    f1 = L.shift_bilinear(f0, *L.TRUE_SHIFT)
    ex, ey = L.estimate_shift(f0, f1)
    checks.append(("known shift (2.4, 1.3) pixels per frame is recovered within 0.1 pixel",
                   abs(ex - 2.4) < 0.1 and abs(ey - 1.3) < 0.1))
    rng = np.random.default_rng(7)
    ok = True
    for tr in [(2.4, 1.3), (-1.7, 0.6), (0.35, -2.85)]:
        a = np.clip(f0 + rng.normal(0, 0.5, f0.shape), 0, None)
        b = np.clip(L.shift_bilinear(f0, *tr) + rng.normal(0, 0.5, f0.shape), 0, None)
        sx, sy = L.estimate_shift(a, b)
        ok &= abs(sx - tr[0]) < 0.1 and abs(sy - tr[1]) < 0.1
    checks.append(("shifts recovered within 0.1 pixel even with 0.5 mm/h seeded noise on both frames", bool(ok)))
    px, py = L.xcorr_peak(f0, f1)
    checks.append(("the FFT cross-correlation peak gives the whole-pixel part (2, 1)", (px, py) == (2, 1)))

    # ---- worked example: unit conversion and CSI from counts
    u, v = ex * 12.0, ey * 12.0
    checks.append(("worked example: (2.4, 1.3) px per 5 min is (28.8, 15.6) km/h, speed 32.76 km/h",
                   abs(u - 28.8) < 1.2 and abs(v - 15.6) < 1.2 and abs(np.hypot(28.8, 15.6) - 32.76) < 0.01))
    checks.append(("worked example: in 60 minutes the field moves (28.8, 15.6) km", abs(2.4 * 12 - 28.8) < 1e-9 and abs(1.3 * 12 - 15.6) < 1e-9))
    checks.append(("worked example: 80 hits, 10 misses, 10 false alarms give CSI 0.8", abs(L.csi_counts(80, 10, 10) - 0.8) < 1e-12))

    # ---- scores on the 60-minute forecast
    obs = L.make_field(L.moved_cells(L.CELLS, *L.TRUE_SHIFT, 12))
    pers, adv = f0, L.advect_forecast(f0, ex, ey, 12)
    checks.append(("60 min persistence: RMSE 5.83 mm/h, CSI 0.074",
                   abs(L.rmse(pers, obs) - 5.83) < 0.01 and abs(L.csi(pers, obs) - 0.074) < 0.002))
    checks.append(("60 min advection: RMSE 0.03 mm/h, CSI 0.996",
                   abs(L.rmse(adv, obs) - 0.03) < 0.01 and abs(L.csi(adv, obs) - 0.996) < 0.002))
    beats = True
    for k in range(1, 13):
        ob = L.make_field(L.moved_cells(L.CELLS, *L.TRUE_SHIFT, k))
        a = L.advect_forecast(f0, ex, ey, k)
        beats &= L.csi(a, ob) > L.csi(f0, ob) and L.rmse(a, ob) < L.rmse(f0, ob)
    checks.append(("advection beats persistence in CSI and RMSE at every lead from 5 to 60 minutes", bool(beats)))
    checks.append(("a perfect forecast scores CSI 1, RMSE 0, FSS 1", L.csi(obs, obs) == 1.0 and L.rmse(obs, obs) == 0.0 and abs(L.fss(obs, obs, n=5) - 1.0) < 1e-12))
    checks.append(("CSI counts a pixel exactly at the threshold as rain", L.csi(np.array([[5.0]]), np.array([[5.0]])) == 1.0))
    checks.append(("no rain in either picture: CSI and FSS are undefined (nan), not 1 or a crash",
                   np.isnan(L.csi(np.zeros((4, 4)), np.zeros((4, 4)))) and np.isnan(L.fss(np.zeros((4, 4)), np.zeros((4, 4)), n=3))))

    # ---- FSS forgives displacement; box_mean is right
    o = L.make_field(L.CELLS)
    f = L.shift_int(o, 3, 0)
    fs = [L.fss(f, o, n=n) for n in (1, 7, 31)]
    checks.append(("a forecast displaced by 3 pixels: CSI 0.677, FSS 0.807 (n=1), 0.929 (n=7), 0.987 (n=31)",
                   abs(L.csi(f, o) - 0.677) < 0.002 and abs(fs[0] - 0.807) < 0.002 and abs(fs[1] - 0.929) < 0.002 and abs(fs[2] - 0.987) < 0.002))
    checks.append(("FSS grows with the window size", fs[0] < fs[1] < fs[2]))
    a = np.arange(25, dtype=float).reshape(5, 5)
    bm = L.box_mean(a, 3)
    pad = np.pad(a, 1)
    checks.append(("box_mean equals a direct 3 x 3 average with zeros outside",
                   all(abs(bm[i, j] - pad[i:i + 3, j:j + 3].mean()) < 1e-9 for i in range(5) for j in range(5))))

    # ---- two-disc model
    r = L.edge_radius(5.0, 30.0)
    checks.append(("a sigma = 5, 30 mm/h cell has a 5 mm/h edge at 9.47 km", abs(r - 9.465) < 0.001))
    c0 = L.make_field([(40, 50, 5.0, 30.0)])
    good = True
    for d in (0, 2, 5, 9):
        good &= abs(L.csi(L.make_field([(40 + d, 50, 5.0, 30.0)]), c0) - L.disc_csi(r, d)) < 0.02
    checks.append(("the two-disc formula matches the grid CSI for displacements of 0, 2, 5 and 9 pixels", bool(good)))
    checks.append(("two discs: CSI 1 at d = 0, 0 at d = 2r, 0.50 at d = 5 km for r = 9.47 km",
                   L.disc_csi(r, 0.0) == 1.0 and L.disc_csi(r, 2 * r) == 0.0 and abs(L.disc_csi(r, 5.0) - 0.501) < 0.002))
    checks.append(("a cell weaker than the threshold has radius 0", L.edge_radius(5.0, 4.0) == 0.0))
    checks.append(("exercise: (1.5, 0.5) pixels per 5 min is (18, 6) km/h, speed 18.97 km/h",
                   abs(1.5 * 12 - 18.0) < 1e-9 and abs(0.5 * 12 - 6.0) < 1e-9 and abs(np.hypot(18.0, 6.0) - 18.97) < 0.01))
    checks.append(("two discs: only the ratio d/r matters (doubling both leaves CSI unchanged)", abs(L.disc_csi(2 * r, 10.0) - L.disc_csi(r, 5.0)) < 1e-12))
    checks.append(("design: wind error 5 km/h over 60 min is 5 km (CSI 0.50); over 30 min 10 km/h is also 5 km",
                   abs(5.0 * 60 / 60 - 5.0) < 1e-12 and abs(10.0 * 30 / 60 - 5.0) < 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
